In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


In [3]:


from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

# =========================
# ✅ threshold 설정 (여기만 바꾸면 전체 적용)
# =========================
THR = 0.6

# =========================
# 데이터 로드
# =========================
df = pd.read_csv("new_flight_weather_merged.csv")
print("✅ CSV 로드:", len(df))

# =========================
# 시간 파생 변수
# =========================
df["departure_datetime"] = pd.to_datetime(df["departure_datetime"], errors="coerce")
df = df[df["departure_datetime"].notna()].copy()

# ⏱ 시간 파생
df["dep_hour"] = df["departure_datetime"].dt.hour
df["dep_minute"] = df["departure_datetime"].dt.minute   # ✅ 분 추가
df["dep_weekday"] = df["departure_datetime"].dt.weekday
df["is_weekend"] = df["dep_weekday"].isin([5, 6]).astype(int)

# =========================
# ✅ 도착지 코드화: arrival_code (cat_cols로 쓸 예정)
# =========================
if "arrival_code" in df.columns:
    df["arrival_code"] = (
        pd.to_numeric(df["arrival_code"], errors="coerce")
          .fillna(-1)
          .astype(int)
    )
else:
    if "도착지" not in df.columns:
        raise KeyError("df에 'arrival_code' 또는 '도착지' 컬럼이 없습니다.")
    df["arrival_code"] = (
        pd.to_numeric(df["도착지"], errors="coerce")
          .fillna(-1)
          .astype(int)
    )

# =========================
# 컬럼 정의
# =========================
num_cols = [
    "기온(°C)",
    "풍속_ms",
    "dep_hour",
    "dep_minute",
    "dep_weekday",
    "is_weekend"
]
num_cols = [c for c in num_cols if c in df.columns]


cat_cols = [
    "항공사",
    "출발지",
    "arrival_code",
    "flight_type"
]
cat_cols = [c for c in cat_cols if c in df.columns]

X_cols = num_cols + cat_cols

print("✅ num_cols:", num_cols)
print("✅ cat_cols:", cat_cols)

# =========================
# 시간 기준 Train/Test 분리
# =========================
df = df.sort_values("departure_datetime")
split_date = df["departure_datetime"].quantile(0.8)

train_df = df[df["departure_datetime"] <= split_date]
test_df  = df[df["departure_datetime"] > split_date]

X_train = train_df[X_cols]
y_train = train_df["is_delay"]

X_test  = test_df[X_cols]
y_test  = test_df["is_delay"]

print("Train:", len(train_df), "Test:", len(test_df))

# =====================================================
# Logistic Regression (One-hot)
# =====================================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

logistic = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

logistic.fit(X_train, y_train)

log_prob = logistic.predict_proba(X_test)[:, 1]
log_pred = (log_prob >= THR).astype(int)

print(f"\n📊 Logistic (threshold={THR})")
print(classification_report(y_test, log_pred))
print("ROC-AUC:", roc_auc_score(y_test, log_prob))
print("PR-AUC :", average_precision_score(y_test, log_prob))


# =====================================================
# XGBoost (One-hot + 트리)
# =====================================================
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model = Pipeline([
    ("prep", preprocessor),
    ("xgb", xgb)
])

xgb_model.fit(X_train, y_train)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_prob >= THR).astype(int)

print(f"\n📊 XGBoost (threshold={THR})")
print(classification_report(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob))
print("PR-AUC :", average_precision_score(y_test, xgb_prob))


# =====================================================
# LightGBM (category)
# =====================================================
from lightgbm import LGBMClassifier

df_lgb = df.copy()
for c in cat_cols:
    df_lgb[c] = df_lgb[c].astype("category")

df_lgb = df_lgb.sort_values("departure_datetime")
train_df_lgb = df_lgb[df_lgb["departure_datetime"] <= split_date]
test_df_lgb  = df_lgb[df_lgb["departure_datetime"] > split_date]

X_train_lgb = train_df_lgb[X_cols]
y_train_lgb = train_df_lgb["is_delay"]

X_test_lgb  = test_df_lgb[X_cols]
y_test_lgb  = test_df_lgb["is_delay"]

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,

    class_weight="balanced",
    objective="binary",
    metric="aucpr",
    random_state=42,
    n_jobs=-1
)

lgbm.fit(
    X_train_lgb,
    y_train_lgb,
    categorical_feature=cat_cols
)

lgb_prob = lgbm.predict_proba(X_test_lgb)[:, 1]
lgb_pred = (lgb_prob >= THR).astype(int)

print(f"\n📊 LightGBM (threshold={THR})")
print(classification_report(y_test_lgb, lgb_pred))
print("ROC-AUC:", roc_auc_score(y_test_lgb, lgb_prob))
print("PR-AUC :", average_precision_score(y_test_lgb, lgb_prob))


# =====================================================
# 성능 요약 표
# =====================================================
summary = pd.DataFrame([
    ["Logistic", roc_auc_score(y_test, log_prob), average_precision_score(y_test, log_prob)],
    ["XGBoost",  roc_auc_score(y_test, xgb_prob), average_precision_score(y_test, xgb_prob)],
    ["LightGBM", roc_auc_score(y_test_lgb, lgb_prob), average_precision_score(y_test_lgb, lgb_prob)]
], columns=["Model", "ROC-AUC", "PR-AUC"])

summary


C:\Users\Admin\AppData\Local\Temp\ipykernel_3844\3699968045.py:15: DtypeWarning: Columns (31,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("new_flight_weather_merged.csv")


✅ CSV 로드: 2836872
✅ num_cols: ['기온(°C)', '풍속_ms', 'dep_hour', 'dep_minute', 'dep_weekday', 'is_weekend']
✅ cat_cols: ['항공사', '출발지', 'arrival_code', 'flight_type']
Train: 2269498 Test: 567374

📊 Logistic (threshold=0.6)
              precision    recall  f1-score   support

           0       0.78      0.83      0.80    421129
           1       0.40      0.32      0.35    146245

    accuracy                           0.70    567374
   macro avg       0.59      0.58      0.58    567374
weighted avg       0.68      0.70      0.69    567374

ROC-AUC: 0.6592586863765935
PR-AUC : 0.3728013256426406

📊 XGBoost (threshold=0.6)
              precision    recall  f1-score   support

           0       0.79      0.84      0.82    421129
           1       0.44      0.36      0.40    146245

    accuracy                           0.72    567374
   macro avg       0.62      0.60      0.61    567374
weighted avg       0.70      0.72      0.71    567374

ROC-AUC: 0.6963783021677618
PR-AUC : 0.43524

,Model,ROC-AUC,PR-AUC
0,Logistic,0.659259,0.372801
1,XGBoost,0.696378,0.435246
2,LightGBM,0.728668,0.482026


In [4]:
# =====================================================
# ✅ 모델 학습/평가/비교 (변수 덮어쓰기 방지 버전)
# - Linear / Ridge / Log-Ridge / LightGBM Raw / LightGBM Log
# - summary_df, preds 생성
# - LightGBM이 제일 잘 맞는 걸 보여주는 비교 시각화 3종 포함
# =====================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

# ✅ 폰트
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

def RMSE(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# =====================================================
# 1) 모델 정의
# =====================================================

lin_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression())
])

ridge_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", Ridge(alpha=1.0))
])

lgbm_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

# =====================================================
# 2) 학습 + 예측 (✅ 변수명 분리)
# =====================================================

# (1) Linear
lin_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)

# (2) Ridge
ridge_reg.fit(X_train, y_train)
y_pred_ridge = ridge_reg.predict(X_test)

# (3) Log-Ridge
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

ridge_reg.fit(X_train, y_train_log)
y_pred_log_ridge_log = ridge_reg.predict(X_test)     # log-space 예측
y_pred_log_ridge = np.expm1(y_pred_log_ridge_log)    # 분 단위로 원복

# (4) LightGBM Raw
lgbm_reg.fit(X_train, y_train)
y_pred_lgbm_raw = lgbm_reg.predict(X_test)

# (5) LightGBM Log Target
lgbm_reg.fit(X_train, y_train_log)
y_pred_lgbm_logspace = lgbm_reg.predict(X_test)
y_pred_lgbm_log = np.expm1(y_pred_lgbm_logspace)     # 분 단위 원복

# =====================================================
# 3) 성능 요약표 (summary_df)
# =====================================================

summary_df = pd.DataFrame({
    "모델": [
        "Linear Regression",
        "Ridge Regression",
        "Log-Ridge Regression",
        "LightGBM (Raw Target)",
        "LightGBM (Log Target)"
    ],
    "MAE (분)": [
        mean_absolute_error(y_test, y_pred_lin),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_log_ridge),
        mean_absolute_error(y_test, y_pred_lgbm_raw),
        mean_absolute_error(y_test, y_pred_lgbm_log)
    ],
    "RMSE (분)": [
        RMSE(y_test, y_pred_lin),
        RMSE(y_test, y_pred_ridge),
        RMSE(y_test, y_pred_log_ridge),
        RMSE(y_test, y_pred_lgbm_raw),
        RMSE(y_test, y_pred_lgbm_log)
    ]
}).sort_values("MAE (분)").reset_index(drop=True)

display(summary_df)

# =====================================================
# 3-1) 지표 비교 테이블 (PPT / 리포트용)
# =====================================================

metrics_df = pd.DataFrame({
    "모델": summary_df["모델"],
    "MAE (분)": summary_df["MAE (분)"].round(2),
    "RMSE (분)": summary_df["RMSE (분)"].round(2),
})

# 베스트 모델 표시용 컬럼
metrics_df["비고"] = ""
metrics_df.loc[metrics_df["모델"] == best_name, "비고"] = "✅ Best"

display(metrics_df)


# =====================================================
# 4) 모델별 예측값 모음 (preds)  ✅ 여기서부터 비교 시각화 가능
# =====================================================

preds = {
    "Linear Regression": y_pred_lin,
    "Ridge Regression": y_pred_ridge,
    "Log-Ridge Regression": y_pred_log_ridge,
    "LightGBM (Raw Target)": y_pred_lgbm_raw,
    "LightGBM (Log Target)": y_pred_lgbm_log
}

best_name = summary_df.loc[0, "모델"]

print("✅ best model =", best_name)

# =====================================================
# 5) (시각화 1) 모델 랭킹 막대: MAE (베스트 모델만 강조)
# =====================================================

colors = ["tab:orange" if m == best_name else "lightgray" for m in summary_df["모델"]]

plt.figure(figsize=(9,4))
plt.barh(summary_df["모델"], summary_df["MAE (분)"], color=colors)
plt.gca().invert_yaxis()
plt.title(f"모델별 MAE 비교 (1등: {best_name})")
plt.xlabel("MAE(분) (낮을수록 좋음)")
for i, v in enumerate(summary_df["MAE (분)"]):
    plt.text(v, i, f"  {v:.1f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

# =====================================================
# 6) (시각화 2) 오차 허용치별 적중률(Accuracy@k): |오차| ≤ k
#    -> PPT에서 “LightGBM이 더 자주 맞춘다”를 제일 직관적으로 보여줌
# =====================================================

y_true = np.asarray(y_test)
thresholds = np.arange(0, 61, 5)  # 0~60분 (원하면 120까지 늘리기)

plt.figure(figsize=(10,4))
for name, p in preds.items():
    p = np.asarray(p)
    abs_err = np.abs(p - y_true)
    acc = [(abs_err <= t).mean() * 100 for t in thresholds]

    if name == best_name:
        plt.plot(thresholds, acc, marker="o", linewidth=3, label=f"{name} (베스트)")
    else:
        plt.plot(thresholds, acc, alpha=0.35, label=name)

plt.title("오차 허용치(k분)별 적중률 비교 (높을수록 좋음)")
plt.xlabel("허용 오차 k(분)")
plt.ylabel("적중률(%)  =  |예측-실제| ≤ k")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

# =====================================================
# 7) (시각화 3) 지연 구간별 MAE: 긴 지연에서도 누가 덜 무너지는지
# =====================


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015123 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 763
[LightGBM] [Info] Number of data points in the train set: 2269498, number of used features: 152
[LightGBM] [Info] Start training from score 0.148124


C:\Users\Admin\anaconda3\envs\4vector\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013036 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 763
[LightGBM] [Info] Number of data points in the train set: 2269498, number of used features: 152
[LightGBM] [Info] Start training from score 0.102672


C:\Users\Admin\anaconda3\envs\4vector\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,모델,MAE (분),RMSE (분)
0,LightGBM (Log Target),0.290640,0.444223
1,Log-Ridge Regression,0.299581,0.453143
2,LightGBM (Raw Target),0.303957,0.431175
3,Ridge Regression,0.314990,0.440793
4,Linear Regression,0.315011,0.440828


NameError: name 'best_name' is not defined